In [0]:
# Load Bronze tables into Spark DataFrames

bronze_orders = spark.table("dbacademy.retailflow.bronze_orders")

bronze_order_items = spark.table("dbacademy.retailflow.bronze_order_items")

bronze_products = spark.table("dbacademy.retailflow.bronze_products")

bronze_stores = spark.table("dbacademy.retailflow.bronze_stores")

print("All Bronze tables loaded successfully!")

All Bronze tables loaded successfully!


In [0]:
print("Orders:", bronze_orders.count())
print("Order Items:", bronze_order_items.count())
print("Products:", bronze_products.count())
print("Stores:", bronze_stores.count())

Orders: 503
Order Items: 1000
Products: 50
Stores: 10


In [0]:
from pyspark.sql.functions import col, to_date

# Check duplicate orders
duplicate_count = (
    bronze_orders.count()
    - bronze_orders.dropDuplicates(["order_id"]).count()
)

# Check missing customer IDs
missing_customer_count = bronze_orders.filter(
    col("customer_id").isNull()
).count()

print("Total Bronze Orders:", bronze_orders.count())
print("Duplicate Orders:", duplicate_count)
print("Missing Customer IDs:", missing_customer_count)

Total Bronze Orders: 503
Duplicate Orders: 3
Missing Customer IDs: 13


##Transform Bronze Orders into Silver Orders

In [0]:
from pyspark.sql.functions import col, to_date

# Step 1: Remove duplicate orders
silver_orders = bronze_orders.dropDuplicates(["order_id"])

# Step 2: Remove records with missing customer IDs
silver_orders = silver_orders.filter(
    col("customer_id").isNotNull()
)

# Step 3: Convert columns into appropriate data types
silver_orders = silver_orders \
    .withColumn("order_id", col("order_id").cast("int")) \
    .withColumn("customer_id", col("customer_id").cast("int")) \
    .withColumn("store_id", col("store_id").cast("int")) \
    .withColumn("order_date", to_date(col("order_date")))

print("Orders transformation completed!")

Orders transformation completed!


##Verify the transformation

In [0]:
print("Silver Orders Count:", silver_orders.count())

print("Remaining Duplicate Orders:",
      silver_orders.count() -
      silver_orders.dropDuplicates(["order_id"]).count())

print("Remaining Missing Customer IDs:",
      silver_orders.filter(
          col("customer_id").isNull()
      ).count())

silver_orders.show(10, truncate=False)

silver_orders.printSchema()

Silver Orders Count: 487
Remaining Duplicate Orders: 0
Remaining Missing Customer IDs: 0
+--------+-----------+--------+----------+--------------------------+
|order_id|customer_id|store_id|order_date|ingestion_timestamp       |
+--------+-----------+--------+----------+--------------------------+
|1       |1066       |8       |2026-01-18|2026-11-14 06:23:22.323674|
|2       |1097       |1       |2026-01-12|2026-11-14 06:23:22.323674|
|3       |1020       |3       |2026-01-15|2026-11-14 06:23:22.323674|
|4       |1055       |10      |2026-03-29|2026-11-14 06:23:22.323674|
|5       |1050       |7       |2026-01-09|2026-11-14 06:23:22.323674|
|6       |1060       |9       |2026-03-18|2026-11-14 06:23:22.323674|
|7       |1071       |1       |2026-02-02|2026-11-14 06:23:22.323674|
|8       |1093       |2       |2026-03-29|2026-11-14 06:23:22.323674|
|9       |1069       |5       |2026-03-29|2026-11-14 06:23:22.323674|
|10      |1044       |2       |2026-03-24|2026-11-14 06:23:22.323674|
+

##Save the Silver Orders Delta Table


In [0]:
silver_orders.write \
    .format("delta") \
        .mode("overwrite") \
            .saveAsTable("dbacademy.retailflow.silver_orders")

print("Silver Orders table created Successfully!")

Silver Orders table created Successfully!


In [0]:
%sql
SELECT COUNT(*) AS silver_orders_count
FROM dbacademy.retailflow.silver_orders;

silver_orders_count
487


In [0]:
%sql
SELECT *
FROM dbacademy.retailflow.silver_orders
LIMIT 10;

order_id,customer_id,store_id,order_date,ingestion_timestamp
12,1001,5,2026-02-28,2026-11-14T06:23:22.323Z
18,1077,6,2026-01-01,2026-11-14T06:23:22.323Z
38,1063,4,2026-03-27,2026-11-14T06:23:22.323Z
67,1024,2,2026-01-11,2026-11-14T06:23:22.323Z
70,1075,10,2026-02-01,2026-11-14T06:23:22.323Z
93,1088,5,2026-01-27,2026-11-14T06:23:22.323Z
161,1007,9,2026-03-01,2026-11-14T06:23:22.323Z
186,1029,5,2026-03-04,2026-11-14T06:23:22.323Z
190,1069,1,2026-03-21,2026-11-14T06:23:22.323Z
218,1055,6,2026-03-10,2026-11-14T06:23:22.323Z


## Clean the Order Items data


In [0]:
from pyspark.sql.functions import col

# Count invalid quantities
invalid_quantity_count = bronze_order_items.filter(
    col("quantity") <= 0
).count()

# Count duplicate order item IDs
duplicate_item_count = (
    bronze_order_items.count()
    - bronze_order_items.dropDuplicates(["order_item_id"]).count()
)

print("Total Order Items:", bronze_order_items.count())
print("Invalid Quantities:", invalid_quantity_count)
print("Duplicate Order Items:", duplicate_item_count)

Total Order Items: 1000
Invalid Quantities: 10
Duplicate Order Items: 0


## Transform the Order Items data


In [0]:
from pyspark.sql.functions import col

# Step 1: Remove duplicate order items
silver_order_items = bronze_order_items.dropDuplicates(
    ["order_item_id"]
)

# Step 2: Remove invalid quantities
silver_order_items = silver_order_items.filter(
    col("quantity") > 0
)

# Step 3: Standardize column data types
silver_order_items = silver_order_items \
    .withColumn("order_item_id", col("order_item_id").cast("int")) \
    .withColumn("order_id", col("order_id").cast("int")) \
    .withColumn("product_id", col("product_id").cast("int")) \
    .withColumn("quantity", col("quantity").cast("int"))

print("Order Items transformation completed!")

Order Items transformation completed!


### Verify the transformation


In [0]:
print("Silver Order Items:", silver_order_items.count())

print(
    "Remaining Invalid Quantities:",
    silver_order_items.filter(col("quantity") <= 0).count()
)

print(
    "Remaining Duplicate Items:",
    silver_order_items.count()
    - silver_order_items.dropDuplicates(["order_item_id"]).count()
)

silver_order_items.show(10, truncate=False)

silver_order_items.printSchema()

Silver Order Items: 990
Remaining Invalid Quantities: 0
Remaining Duplicate Items: 0
+-------------+--------+----------+--------+--------------------------+
|order_item_id|order_id|product_id|quantity|ingestion_timestamp       |
+-------------+--------+----------+--------+--------------------------+
|1            |57      |19        |4       |2026-11-14 06:23:27.216988|
|2            |250     |34        |5       |2026-11-14 06:23:27.216988|
|3            |24      |15        |3       |2026-11-14 06:23:27.216988|
|4            |307     |4         |4       |2026-11-14 06:23:27.216988|
|5            |105     |20        |1       |2026-11-14 06:23:27.216988|
|6            |393     |9         |2       |2026-11-14 06:23:27.216988|
|7            |149     |21        |3       |2026-11-14 06:23:27.216988|
|8            |4       |32        |1       |2026-11-14 06:23:27.216988|
|9            |90      |9         |4       |2026-11-14 06:23:27.216988|
|10           |273     |46        |4       |2026-11

In [0]:
silver_order_items.write \
    .format("delta") \
        .mode("overwrite") \
            .saveAsTable("dbacademy.retailflow.silver_order_items")

print("Silver Order Items table created Successfully!")

Silver Order Items table created Successfully!


In [0]:
%sql
SELECT COUNT(*) AS total_silver_order_items
FROM dbacademy.retailflow.silver_order_items;

total_silver_order_items
990


## Clean the Products data


In [0]:
from pyspark.sql.functions import col

# Check missing product categories
missing_category_count = bronze_products.filter(
    col("category").isNull()
).count()

# Check invalid product prices
invalid_price_count = bronze_products.filter(
    col("price").isNull() | (col("price") <= 0)
).count()

# Check duplicate product IDs
duplicate_product_count = (
    bronze_products.count()
    - bronze_products.dropDuplicates(["product_id"]).count()
)

print("Total Products:", bronze_products.count())
print("Missing Categories:", missing_category_count)
print("Invalid Prices:", invalid_price_count)
print("Duplicate Products:", duplicate_product_count)

Total Products: 50
Missing Categories: 11
Invalid Prices: 0
Duplicate Products: 0


### Transform Products


In [0]:
from pyspark.sql.functions import col, trim, when

# Step 1: Remove duplicate products
silver_products = bronze_products.dropDuplicates(["product_id"])

# Step 2: Replace missing or blank categories
silver_products = silver_products.withColumn(
    "category",
    when(
        col("category").isNull() | (trim(col("category")) == ""),
        "Unknown"
    ).otherwise(trim(col("category")))
)

# Step 3: Standardize data types
silver_products = silver_products \
    .withColumn("product_id", col("product_id").cast("int")) \
    .withColumn("product_name", trim(col("product_name"))) \
    .withColumn("price", col("price").cast("double"))

print("Products transformation completed!")

Products transformation completed!


### Verify Products

In [0]:
print("Silver Products:", silver_products.count())

print(
    "Missing Categories:",
    silver_products.filter(col("category").isNull()).count()
)

print(
    "Invalid Prices:",
    silver_products.filter(
        col("price").isNull() | (col("price") <= 0)
    ).count()
)

silver_products.show(10, truncate=False)
silver_products.printSchema()

Silver Products: 50
Missing Categories: 0
Invalid Prices: 0
+----------+------------+-----------+-------+--------------------------+
|product_id|product_name|category   |price  |ingestion_timestamp       |
+----------+------------+-----------+-------+--------------------------+
|1         |Product_1   |Electronics|222.55 |2026-11-14 06:23:30.202387|
|2         |Product_2   |Groceries  |1299.97|2026-11-14 06:23:30.202387|
|3         |Product_3   |Clothing   |3708.71|2026-11-14 06:23:30.202387|
|4         |Product_4   |Unknown    |526.0  |2026-11-14 06:23:30.202387|
|5         |Product_5   |Furniture  |255.74 |2026-11-14 06:23:30.202387|
|6         |Product_6   |Electronics|1171.33|2026-11-14 06:23:30.202387|
|7         |Product_7   |Unknown    |3049.89|2026-11-14 06:23:30.202387|
|8         |Product_8   |Unknown    |1074.3 |2026-11-14 06:23:30.202387|
|9         |Product_9   |Unknown    |2155.65|2026-11-14 06:23:30.202387|
|10        |Product_10  |Furniture  |2987.4 |2026-11-14 06:23:30

### Save the product Table

In [0]:
silver_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbacademy.retailflow.silver_products")

print("Silver Products table created successfully!")

Silver Products table created successfully!


In [0]:
%sql
SELECT COUNT(*) FROM dbacademy.retailflow.silver_products;

COUNT(*)
50


## Clean the Stores data

In [0]:
from pyspark.sql.functions import col

print("Total Stores:", bronze_stores.count())

print(
    "Duplicate Store IDs:",
    bronze_stores.count()
    - bronze_stores.dropDuplicates(["store_id"]).count()
)

print(
    "Missing Store Names:",
    bronze_stores.filter(col("store_name").isNull()).count()
)

print(
    "Missing Cities:",
    bronze_stores.filter(col("city").isNull()).count()
)

Total Stores: 10
Duplicate Store IDs: 0
Missing Store Names: 0
Missing Cities: 0


### Transform Stores data

In [0]:
from pyspark.sql.functions import col, trim

# Step 1: Remove duplicate store records
silver_stores = bronze_stores.dropDuplicates(["store_id"])

# Step 2: Standardize column data types and text
silver_stores = silver_stores \
    .withColumn("store_id", col("store_id").cast("int")) \
    .withColumn("store_name", trim(col("store_name"))) \
    .withColumn("city", trim(col("city")))

print("Stores transformation completed!")

Stores transformation completed!


### Verify Stores

In [0]:
print("Silver Stores:", silver_stores.count())

silver_stores.show(truncate=False)
silver_stores.printSchema()

Silver Stores: 10
+--------+----------+----------+--------------------------+
|store_id|store_name|city      |ingestion_timestamp       |
+--------+----------+----------+--------------------------+
|1       |Store_1   |Chennai   |2026-11-14 06:23:33.178644|
|2       |Store_2   |Bangalore |2026-11-14 06:23:33.178644|
|3       |Store_3   |Hyderabad |2026-11-14 06:23:33.178644|
|4       |Store_4   |Mumbai    |2026-11-14 06:23:33.178644|
|5       |Store_5   |Delhi     |2026-11-14 06:23:33.178644|
|6       |Store_6   |Pune      |2026-11-14 06:23:33.178644|
|7       |Store_7   |Coimbatore|2026-11-14 06:23:33.178644|
|8       |Store_8   |Kochi     |2026-11-14 06:23:33.178644|
|9       |Store_9   |Kolkata   |2026-11-14 06:23:33.178644|
|10      |Store_10  |Jaipur    |2026-11-14 06:23:33.178644|
+--------+----------+----------+--------------------------+

root
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- ingestion_ti

### save the store table

In [0]:
silver_stores.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbacademy.retailflow.silver_stores")

print("Silver Stores table created successfully!")

Silver Stores table created successfully!


In [0]:
%sql
SELECT COUNT(*) FROM dbacademy.retailflow.silver_stores;

COUNT(*)
10


In [0]:
%sql
show tables in dbacademy.retailflow;

database,tableName,isTemporary
retailflow,bronze_order_items,false
retailflow,bronze_orders,false
retailflow,bronze_products,false
retailflow,bronze_stores,false
retailflow,silver_order_items,false
retailflow,silver_orders,false
retailflow,silver_products,false
retailflow,silver_stores,false


In [0]:
%sql
SELECT 'silver_orders' AS table_name, COUNT(*) AS total_records
FROM dbacademy.retailflow.silver_orders

UNION ALL

SELECT 'silver_order_items', COUNT(*)
FROM dbacademy.retailflow.silver_order_items

UNION ALL

SELECT 'silver_products', COUNT(*)
FROM dbacademy.retailflow.silver_products

UNION ALL

SELECT 'silver_stores', COUNT(*)
FROM dbacademy.retailflow.silver_stores;

table_name,total_records
silver_orders,487
silver_order_items,990
silver_products,50
silver_stores,10


In [0]:
%sql

SELECT COUNT(*) AS invalid_quantity_records
FROM dbacademy.retailflow.silver_order_items
WHERE quantity <= 0;

invalid_quantity_records
0


In [0]:
%sql

SELECT category, COUNT(*) AS product_count
FROM dbacademy.retailflow.silver_products
GROUP BY category
ORDER BY product_count DESC;

category,product_count
Groceries,11
Unknown,11
Electronics,10
Clothing,9
Furniture,9
